In [2]:
import numpy as np
from qiskit_qaoa.utils.hamiltonian_utils import get_normalised_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples

In [163]:
filename = 'test_N2_W2'
data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'

_, hamiltonian, _, ising_offset, _, ham_norm = get_normalised_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits


# keys = [np.binary_repr(x, num_qubits) for x in range(2**num_qubits)]

# evals = ham_norm * (evaluate_sparse_pauli_samples(keys, hamiltonian) + ising_offset)


In [151]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
num_qubits = 3

prob = 1 / (2 * 3)
theta = 2 * np.arcsin(np.sqrt(prob))
init_angles = theta * np.ones((num_qubits,))
print(init_angles)
betas = ParameterVector('β', 1)


init = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    init.ry(init_angles[i], i)
    
mixer = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    mixer.ry(-init_angles[i], i)
    mixer.rz(2*betas[0], i)
    mixer.ry(init_angles[i], i)

[0.84106867 0.84106867 0.84106867]


In [159]:
from qiskit_aer import AerSimulator
sim = AerSimulator(method='statevector')
circ = QuantumCircuit(num_qubits)
circ.compose(init, inplace=True)

circ2 = circ.compose(mixer, inplace=False)
circ3 = circ2.compose(mixer, inplace=False)

circ.save_statevector()
circ2.save_statevector()
circ3.save_statevector()


circ2.assign_parameters([2*np.pi + 0.5], inplace=True)
circ3.assign_parameters([0.5], inplace=True)

res = sim.run(circ).result()
res2 = sim.run(circ2).result()
res3 = sim.run(circ3).result()

In [160]:
sv = np.asarray(res.results[0].data.statevector)
sv2 = np.asarray(res2.results[0].data.statevector)
sv3 = np.asarray(res3.results[0].data.statevector)

In [161]:
sv2/sv, sv3/sv2

(array([0.0707372-0.99749499j, 0.0707372-0.99749499j,
        0.0707372-0.99749499j, 0.0707372-0.99749499j,
        0.0707372-0.99749499j, 0.0707372-0.99749499j,
        0.0707372-0.99749499j, 0.0707372-0.99749499j]),
 array([0.0707372-0.99749499j, 0.0707372-0.99749499j,
        0.0707372-0.99749499j, 0.0707372-0.99749499j,
        0.0707372-0.99749499j, 0.0707372-0.99749499j,
        0.0707372-0.99749499j, 0.0707372-0.99749499j]))

In [162]:
np.exp(1j * 0.5 * -num_qubits)

np.complex128(0.0707372016677029-0.9974949866040544j)

In [54]:
qc = QuantumCircuit(1)
qc.rz(-1, 0)
qc.save_statevector()
sim.run(qc).result().results[0].data.statevector


Statevector([0.87758256+0.47942554j, 0.        +0.j        ],
            dims=(2,))


In [55]:
np.exp(0.5j)

np.complex128(0.8775825618903728+0.479425538604203j)

In [60]:
qc = QuantumCircuit(1)
qc.rz(-1, 0)
qc.ry(0.84107, 0)
qc.save_statevector()
zy = np.asarray(sim.run(qc).result().results[0].data.statevector)

In [59]:
qc = QuantumCircuit(1)
qc.ry(0.84107, 0)
qc.save_statevector()
yonly = np.asarray(sim.run(qc).result().results[0].data.statevector)

In [63]:
zy, yonly, zy/yonly

(array([0.80111937+0.43765351j, 0.35827211+0.19572495j]),
 array([0.91287066+0.j, 0.4082489 +0.j]),
 array([0.87758256+0.47942554j, 0.87758256+0.47942554j]))

In [172]:
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import transpile

cost = QuantumCircuit(num_qubits)
cost.h(range(num_qubits))
g = ParameterVector('g', 1)
cost.append(PauliEvolutionGate(hamiltonian, time=g[0]), range(num_qubits))
cost.save_statevector()
c1 = cost.assign_parameters([2*np.pi + 0.5], inplace=False)
c2 = cost.assign_parameters([0.5], inplace=False)
tc1 = transpile(c1, sim)
tc2 = transpile(c2, sim)
res1 = sim.run(tc1).result()
res2 = sim.run(tc2).result()
sv1 = np.asarray(res1.results[0].data.statevector)
sv2 = np.asarray(res2.results[0].data.statevector)


In [174]:
sv1[:5], sv2[:5], sv2[:5]/sv1[:5]

(array([0.02927783-0.05521828j, 0.06228346-0.00519812j,
        0.06228346-0.00519812j, 0.02927783-0.05521828j,
        0.06228346-0.00519812j]),
 array([-0.01253509+0.06123007j, -0.05829607+0.02253484j,
        -0.05829607+0.02253484j, -0.01253509+0.06123007j,
        -0.05829607+0.02253484j]),
 array([-0.95949297+0.28173256j, -0.95949297+0.28173256j,
        -0.95949297+0.28173256j, -0.95949297+0.28173256j,
        -0.95949297+0.28173256j]))